# latent_lab: read → operate → render in one notebook

We edit the latent $z = \varphi(x)$ of a frozen text encoder through an operator on structured states:

| piece | symbol | here |
|---|---|---|
| encoder | $\varphi$ | `Encoder("nomic", 128)`, also `qwen3`, `embeddinggemma` at several widths |
| reader | $\psi(s \mid z)$ | `Reader(columns)`: a linear or MLP probe |
| operator | $\omega_S$ | any function `states DataFrame -> target states DataFrame`, e.g. `LinearSCM(...).do(X=3)` |
| renderer | $\rho$ | `Renderer(kind, columns)`: writes $s'$ back into $z$ |

Reader and renderer see only **factual** $(z, s)$, so operators can be swapped freely. Paired counterfactual latents $\varphi(x')$ are used for evaluation only. Notation: `docs/contribution.md`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from latent_lab import (
    ENCODERS, RENDERERS, ConstantRenderer, Encoder, FlowRenderer, GroupOTRenderer, IdentityRenderer,
    LinearSCM, Pipeline, Reader, StateEffectRenderer, evaluate,
)

ROOT = Path.cwd().parent if Path.cwd().name == "latent_lab" else Path.cwd()  # repo root
print("encoders: ", ENCODERS)
print("renderers:", {name: cls.__name__ for name, cls in RENDERERS.items()})

## 1. Data

Any table of discrete states, one row per unit and indexed by unit id, works. Here: the talent experiment (`exp/sim_talent/`). Country of origin `X`, hidden talent `T`, degree `D` and university tier `U` form the state; the proxies `P L H A` and the outcome `Y` are in the table too.

Note: use `df["T"]`, never `df.T` (that is pandas' transpose).

In [ ]:
factual = pd.read_csv(ROOT / "data/sim_talent/sim_data_factual.csv", index_col="id")
counterfactual = pd.read_csv(ROOT / "data/sim_talent/sim_data_counterfactual.csv", index_col="id")  # under do(X=3)
split = pd.read_csv(ROOT / "data/sim_talent/pair_index.csv", index_col="id")["split"]

STATE = ["X", "T", "D", "U"]
factual.head()

## 2. Encoder φ

`latents("talent")` loads the cached encodings of that experiment's CV texts (no model download): `latents.factual(ids)` gives $\varphi(x)$ and `latents.counterfactual(ids)` gives $\varphi(x')$, which exists for the test units only.

In [ ]:
encoder = Encoder("nomic", 128)          # e.g. Encoder("qwen3", 256), Encoder("embeddinggemma", 768)
latents = encoder.latents("talent")

train_ids = split.index[split == "train"]
test_ids = latents.cf_ids
z_train, s_train = latents.factual(train_ids), factual.loc[train_ids]
z_test, s_test = latents.factual(test_ids), factual.loc[test_ids]
z_cf_test, s_cf_test = latents.counterfactual(test_ids), counterfactual.loc[test_ids]
z_train.shape, z_test.shape

# Live encoding (nomic runs here; qwen3/embeddinggemma need their own venvs):
# encoder.encode(["I hold a Master's degree from ..."])

## 3. Reader ψ

`hidden=0` is a linear probe, `hidden=1` an MLP probe. Two of them: `reader` sits inside the pipeline, `judge` is an independent probe used only to evaluate edited latents.

In [ ]:
reader = Reader(STATE, hidden=0).fit(z_train, s_train)
judge = Reader(STATE, hidden=1).fit(z_train, s_train, seed=1)
pd.DataFrame({"reader": reader.accuracy(z_test, s_test), "judge": judge.accuracy(z_test, s_test)}).round(3)

## 4. Operators ω_S

An operator is any function `states DataFrame -> target states DataFrame`. Causal operators come from a `LinearSCM`.

### The SCM

Each non-root node is `clip(round(Σ weight · parent + ε))`, with `ε ~ Normal(mu, sd)`. These numbers are copied from `exp/sim_talent/talent_sfm.py`, which generated the data. **Editing them changes only the operator, not the data**, so an edited SCM is a misspecified operator.

`scm.counterfactual(states, do, blocked)` follows the three steps:
1. **Abduction:** draw each node's noise from the range consistent with the factual state.
2. **Action:** set the do() values.
3. **Prediction:** recompute the nodes downstream.

`blocked={"D": ["X"]}` makes `D` read the *factual* `X`: the path-specific counterfactual. `scm.do(X=3, blocked=...)` returns that as a function for the pipeline.

In [ ]:
LEVELS = {"X": 4, "T": 3, "D": 3, "U": 3}
PARENTS = {                       # node: {parent: weight}, in topological order; X and T are roots
    "D": {"X": 0.30, "T": 0.60},
    "U": {"X": 0.35, "T": 0.50},
}
NOISE = {"D": (-0.10, 0.50), "U": (-0.15, 0.50)}   # node: (mu, sd)

scm = LinearSCM(LEVELS, PARENTS, NOISE)
wrong_scm = LinearSCM(LEVELS, {**PARENTS, "D": {"T": 0.60}}, NOISE)   # misses the edge X -> D
print(scm)

### Any other operator

A do() value can also be a function of the states, drawn at every call: `scm.do(X=lambda s: ...)` is a **random intervention**. With $X' \sim P(X)$ it erases `X` causally: `D` and `U` follow the new `X`.

Next to the SCM operators, any function works. `relabel X` redraws only the `X` label and leaves `D` and `U` at their factual values: a non-causal "blur" of `X`. `U + 1` is a deterministic rule.

In [ ]:
rng = np.random.default_rng(0)
p_X = factual["X"].value_counts(normalize=True).sort_index().to_numpy()

operators = {
    "total effect": scm.do(X=3),
    "direct effect": scm.do(X=3, blocked={"D": ["X"], "U": ["X"]}),
    "wrong SCM (no X->D)": wrong_scm.do(X=3),
    "random do(X ~ P(X))": scm.do(X=lambda s: rng.choice(4, size=len(s), p=p_X)),
    "relabel X (no SCM)": lambda s: s.assign(X=rng.choice(4, size=len(s), p=p_X)),
    "U + 1": lambda s: s.assign(U=np.minimum(s["U"] + 1, 2)),
}
operators["total effect"](s_test[STATE].head())

In [ ]:
# sanity check: the SCM operator on the true factual states vs. the simulated counterfactual states
pd.DataFrame({
    "operator": operators["total effect"](s_test[STATE])["D"].value_counts(normalize=True),
    "simulated": s_cf_test["D"].value_counts(normalize=True),
}).sort_index().round(3)

## 5. Renderers ρ

One class per kind, all subclasses of `Renderer`. Each is fitted once on the factual training units, and its settings are constructor arguments, e.g. `FlowRenderer(STATE, epochs=50)` or `GroupOTRenderer(STATE, on="X", shrink=0.2)`. `FlowRenderer` takes about half a minute on CPU. A new renderer subclasses `Renderer` and implements `_fit` and `_render`.

In [ ]:
# projected C-sym needs a guard: a linear probe of the talent proxies whose reading must not change
proxy_guard = Reader(["P", "L", "H", "A"]).fit(z_train, s_train)

renderers = {
    "identity": IdentityRenderer(STATE),
    "constant": ConstantRenderer(STATE, on="X"),
    "group_ot": GroupOTRenderer(STATE, on="X", shrink=0.1),
    "state_effect": StateEffectRenderer(STATE),
    "state_effect+guard": StateEffectRenderer(STATE, guard=proxy_guard),
    "flow": FlowRenderer(STATE, epochs=100),
}
for renderer in renderers.values():
    renderer.fit(z_train, s_train)

## 6. Pipeline under do(X=3)

`Pipeline(None, ...)` uses the true factual states, so only the renderer can be wrong. `Pipeline(reader, ...)` reads the states from $z$, as in deployment.

- `gap_closed` compares $z'$ with the paired $\varphi(x')$.
- `acc_*` is the judge's accuracy on $z'$ against the true counterfactual states.
- `real_acc_*` is the judge's accuracy on $\varphi(x')$ itself.

In [ ]:
operator = operators["total effect"]
rows = {}
for name, renderer in renderers.items():
    for setting, read in (("true s", None), ("read s", reader)):
        result = Pipeline(read, operator, renderer)(z_test, s_test)
        rows[(name, setting)] = evaluate(z_test, result.z_prime, z_cf_test, judge, s_cf_test)
pd.DataFrame(rows).T.round(3)

## 7. Exchange the operator, keep ψ and ρ

Only the true do(X=3) has paired texts, so the other operators are judged by **fidelity to the command**: does the judge read the commanded $s'$ in $z'$?

In [ ]:
renderer = renderers["flow"]       # a FlowRenderer
rows = {}
for name, operator in operators.items():
    result = Pipeline(reader, operator, renderer)(z_test)
    rows[name] = judge.accuracy(result.z_prime, result.s_prime)
pd.DataFrame(rows).T.round(3)